# Figure 6A: legacy session and block counts

In [1]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from manuscript_analysis.contract import CLASS_FILES, FEATURE_SETS
from manuscript_analysis.data import canonical_session_name, session_key
from manuscript_analysis.figures.fig_utils import ROOT, report_saved

INPUT_DIR = ROOT / 'input' / 'unsorted'
CSV_PATH = ROOT / 'output' / 'tables' / '6A' / '6A_session_table.csv'
LEGACY_SEED = 42
SPLITS = ['Train', 'Validation', 'Test']
BLOCK_SIZE = 500
MIN_RAW_SPIKES = 5_000
MAX_BLOCKS_PER_SESSION = 250
FIRST_HOUR_IN_SAMPLES = 20_000 * 60 * 60
LEGACY_EXCLUSIONS = {'exp36_ad-co-day150-1_19apr24', 'exp35_ad-co-day150-1_19apr22'}

In [2]:
def eligible_sessions(class_name, pickle_name):
    records = []
    for session, frame in pd.read_pickle(INPUT_DIR / pickle_name).groupby('session'):
        if session_key(session) in LEGACY_EXCLUSIONS or len(frame) < MIN_RAW_SPIKES:
            continue
        first_hour = frame.loc[frame['Sample'].le(FIRST_HOUR_IN_SAMPLES), list(FEATURE_SETS['all'])].dropna()
        blocks = min(len(first_hour) // BLOCK_SIZE, MAX_BLOCKS_PER_SESSION)
        records.append({'Session': canonical_session_name(session), 'Class': class_name, 'Blocks': blocks})
    return records


split_rows = []
for class_name, pickle_name in CLASS_FILES.items():
    train, test = train_test_split(
        eligible_sessions(class_name, pickle_name), test_size=0.20, random_state=LEGACY_SEED
    )
    train, validation = train_test_split(train, test_size=0.20, random_state=LEGACY_SEED)
    for split, records in zip(SPLITS, (train, validation, test)):
        split_rows += [{**record, 'Split': split} for record in records]

split_manifest = pd.DataFrame(split_rows).sort_values(['Split', 'Class', 'Session'], ignore_index=True)
split_manifest

,Session,Class,Blocks,Split
0,EXP13_AD-CO-DAY150-5_NOV24,AD-CO,145,Test
1,EXP20_AD-CO-DAY150-6_NOV24,AD-CO,47,Test
2,EXP6_AD-CO-DAY150-1_NOV24,AD-CO,39,Test
3,EXP71_AD-CO-DAY150-13_NOV24,AD-CO,250,Test
4,EXP78_AD-CO-DAY150-19_NOV24,AD-CO,250,Test
5,EXP8_AD-CO-DAY150-3_NOV24,AD-CO,250,Test
6,EXP99_AD-CO-DAY150-23_NOV24,AD-CO,250,Test
7,EXP11_CTRL-CO-DAY150-6_NOV24,CTRL-CO,13,Test
8,EXP12_CTRL-CO-DAY150-7_NOV24,CTRL-CO,183,Test
9,EXP3_CTRL-CO-DAY150-3_NOV24,CTRL-CO,14,Test


In [3]:
counts = (
    split_manifest.groupby(['Split', 'Class'])
    .agg(Sessions=('Session', 'nunique'), Blocks=('Blocks', 'sum'))
    .unstack('Class')
)
table = pd.DataFrame(
    {f'{c} {m}': counts[(m, c)] for c in CLASS_FILES for m in ('Sessions', 'Blocks')}
).reindex(SPLITS)

expected = pd.DataFrame(
    {
        'CTRL-CO Sessions': [12, 3, 4],
        'CTRL-CO Blocks': [1575, 307, 296],
        'AD-CO Sessions': [19, 5, 7],
        'AD-CO Blocks': [3749, 546, 1231],
    },
    index=pd.Index(SPLITS, name='Split'),
)
pd.testing.assert_frame_equal(table, expected)

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(CSV_PATH)
display(table)
report_saved([CSV_PATH])

,CTRL-CO Sessions,CTRL-CO Blocks,AD-CO Sessions,AD-CO Blocks
Split,,,,
Train,12,1575,19,3749
Validation,3,307,5,546
Test,4,296,7,1231


Saved output\tables\6A\6A_session_table.csv
